# 01 — Exploratory Data Analysis
Overview of all five tables: shape, dtypes, nulls, distributions.

In [ ]:

import sys; sys.path.insert(0, '..')
from pipeline.ingest import load_all
from pipeline.clean import clean_all
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')

raw = load_all()
cleaned = clean_all(raw)


## Dataset Sizes

In [ ]:

for name, df in cleaned.items():
    print(f"{name:15s}  rows={len(df):>10,}  cols={df.shape[1]}")


## Null Analysis

In [ ]:

for name, df in cleaned.items():
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    if len(nulls):
        print(f"\n--- {name} ---")
        print((nulls / len(df) * 100).round(2).to_string())
    else:
        print(f"{name}: no nulls")


## Customer Age Distribution

In [ ]:

fig, ax = plt.subplots(figsize=(8, 4))
cleaned['customers']['age'].hist(bins=30, ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Customer Age Distribution')
ax.set_xlabel('Age'); ax.set_ylabel('Count')
plt.tight_layout(); plt.show()


## Loyalty Tier Distribution

In [ ]:

tier_counts = cleaned['customers']['loyalty_tier'].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(6, 4))
tier_counts.plot(kind='bar', ax=ax, color=['#cd7f32','#c0c0c0','#ffd700'], edgecolor='white')
ax.set_title('Customer Loyalty Tier Distribution')
ax.set_xlabel('Tier'); ax.set_ylabel('Count')
plt.xticks(rotation=0); plt.tight_layout(); plt.show()


## Event Type Breakdown

In [ ]:

ev_counts = cleaned['events']['event_type'].value_counts()
fig, ax = plt.subplots(figsize=(7, 4))
ev_counts.plot(kind='barh', ax=ax, color='coral')
ax.set_title('Event Type Counts'); ax.set_xlabel('Count')
plt.tight_layout(); plt.show()


## Transaction Revenue Distribution

In [ ]:

tx = cleaned['transactions'].dropna(subset=['gross_revenue'])
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
tx['gross_revenue'].clip(-200, 600).hist(bins=50, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Gross Revenue Distribution'); axes[0].set_xlabel('Revenue ($)')
tx['quantity'].value_counts().sort_index().plot(kind='bar', ax=axes[1], color='teal')
axes[1].set_title('Quantity per Transaction'); axes[1].set_xlabel('Quantity')
plt.tight_layout(); plt.show()


## Product Category Distribution

In [ ]:

cat_counts = cleaned['products']['category'].value_counts()
fig, ax = plt.subplots(figsize=(7, 4))
cat_counts.plot(kind='bar', ax=ax, color='mediumpurple', edgecolor='white')
ax.set_title('Products by Category'); ax.set_xlabel('Category'); ax.set_ylabel('Count')
plt.xticks(rotation=30); plt.tight_layout(); plt.show()


## Monthly Transaction Volume

In [ ]:

monthly = cleaned['transactions'].groupby('month').agg(
    orders=('transaction_id', 'count'),
    revenue=('net_revenue', 'sum')
).reset_index()
fig, ax1 = plt.subplots(figsize=(12, 4))
ax2 = ax1.twinx()
ax1.bar(monthly['month'], monthly['orders'], color='steelblue', alpha=0.6, label='Orders')
ax2.plot(monthly['month'], monthly['revenue'], color='crimson', marker='o', lw=2, label='Revenue')
ax1.set_title('Monthly Orders & Revenue')
ax1.set_xlabel('Month'); ax1.set_ylabel('Orders'); ax2.set_ylabel('Net Revenue ($)')
plt.xticks(rotation=45, ha='right')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, labels1+labels2, loc='upper left')
plt.tight_layout(); plt.show()
